# VietTDR vòng 3 — DASR: recognizer + siêu phân giải nhận thức dấu

Một phiên duy nhất (~7.5h, chốt chặn 10h):

| Cell | Việc | Thời gian |
|---|---|---|
| 1–2 | môi trường, nạp code + data vào /tmp | ~5 phút |
| 3 | test + sinh 500k ảnh cho recognizer | ~25 phút |
| 4 | recognizer full (pretrain 15ep + FT 30ep) + eval + LƯU ckpt | ~5.9h |
| 5 | sinh 120k cặp HR/LR + mặt nạ dấu (render kép) | ~20 phút |
| 6 | SR: S0 (chỉ pix) → S1 (+mặt nạ) → S2 (+recognizer) + eval | ~1.5h |
| 7 | đóng gói | 1 phút |

**Khắc phục lỗi mất output**: mọi thứ chạy trong `/tmp/vt`; chỉ vài file
kết quả được chép sang `/kaggle/working`, và checkpoint được chép NGAY
sau từng chặng chứ không đợi cuối phiên.

Add Input 3 nguồn: `viettdr-data`, dataset code mới nhất (tên có `code`),
`vintext-train-images`. GPU **T4 x2**. Internet không cần.

## Bảng thí nghiệm SR

| Run | Loss | Cô lập |
|---|---|---|
| bicubic | – | hạ đáy (in kèm mỗi lần eval real) |
| S0 | Charbonnier | baseline SR |
| S1 | + mặt nạ dấu | ① render kép |
| S2 | + mặt nạ + loss thành phần recognizer | ② phân rã làm prior |

Chỉ số: PSNR / SSIM / **D-PSNR** (trong vùng dấu) trên cặp tổng hợp, và
**word acc trên crop VinText thật cao ≤16px** (bicubic vs SR).

In [ ]:
# Cell 1 — moi truong + dong ho phien
import torch, os, multiprocessing, time
print('GPU   :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else '*** KHONG CO GPU - bat Accelerator T4 ***')
print('CPU   :', multiprocessing.cpu_count(), 'cores')
IS_KAGGLE = os.path.exists('/kaggle')
NPROC = max(2, multiprocessing.cpu_count())
T0 = time.time()
BUDGET_H = 10.0   # qua moc nay thi bo cac run con lai (gioi han cung 12h)
def elapsed_h(): return (time.time() - T0) / 3600
print(f'ngan sach {BUDGET_H}h')

In [ ]:
# Cell 2 — nap code + du lieu vao /tmp (NGOAI /kaggle/working)
import os, glob, shutil, time

SRC, WORK = ('/kaggle/input', '/tmp/vt') if IS_KAGGLE else \
            ('/content/drive/MyDrive', '/content/run')
OUTDIR = '/kaggle/working' if IS_KAGGLE else '.'
os.makedirs(WORK, exist_ok=True)

def find_all(root, name):
    return sorted(glob.glob(f'{root}/**/{name}', recursive=True))

# chon ban code theo NANG LUC (phai co du cac file/co cua vong 3)
cands = find_all(SRC, 'train.py')
good = []
for c in cands:
    root = os.path.dirname(c)
    try:
        t = open(c, encoding='utf-8').read()
        ok = ('--synth-aug' in t and '--init-from' in t
              and os.path.exists(os.path.join(root, 'train_sr.py'))
              and os.path.exists(os.path.join(root, 'tools', 'gen_sr_pairs.py')))
        if ok:
            good.append(c)
    except Exception:
        pass
print('ban code tim thay:')
for c in cands:
    print(('   [DU CO] ' if c in good else '   [CU]    ') + os.path.dirname(c))
assert good, 'Khong ban code nao co du file SR -> upload VietTDR.zip moi'
code_root = os.path.dirname(good[-1])
for item in ['viettdr', 'tools', 'tests', 'fonts', 'assets', 'train.py',
             'eval.py', 'train_sr.py', 'eval_sr.py', 'charset_vintext.txt']:
    s, d = os.path.join(code_root, item), os.path.join(WORK, item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    elif os.path.isfile(s):
        shutil.copy(s, d)

gt = None
for h in find_all(SRC, 'train_gt.jsonl'):
    if 'vintext' in h.lower():
        gt = h
        break
assert gt, 'Khong tim thay train_gt.jsonl'
data_dst = os.path.join(WORK, 'data', 'vintext_words')
if not os.path.exists(os.path.join(data_dst, 'train_gt.jsonl')):
    t0 = time.time()
    os.makedirs(os.path.dirname(data_dst), exist_ok=True)
    shutil.copytree(os.path.dirname(gt), data_dst)
    print(f'copy du lieu -> /tmp: {time.time() - t0:.0f}s')

os.chdir(WORK)
print('code :', code_root)
print('font :', len(glob.glob(f'{WORK}/fonts/*.ttf')))
!python tests/test_vietchar.py

In [ ]:
# Cell 3 — sinh 500k anh tong hop cho recognizer (nhu vong 2)
import os, glob, json, sys, random
from collections import Counter

corpus = ['data/vintext_words/train_gt.jsonl']
if os.path.exists('assets/general_dict.txt'):
    corpus.append('assets/general_dict.txt')
corpus = ' '.join(corpus)
bg_hits = glob.glob('/kaggle/input/**/im0001.jpg', recursive=True)
BG = ('--bg-dir ' + os.path.dirname(bg_hits[0])) if bg_hits else ''
print('nen that:', os.path.dirname(bg_hits[0]) if bg_hits else 'KHONG CO')

if not os.path.exists('data/synth/synth_gt.jsonl'):
    !python tools/gen_synth.py --out data/synth --count 500000 \
        --fonts fonts --corpus {corpus} --workers {NPROC} \
        --height-ref data/vintext_words/train_gt.jsonl \
        --height-ref-dir data/vintext_words/train \
        --bg-cache 40 --tone-balance {BG}

lines = open('data/synth/synth_gt.jsonl', encoding='utf-8').read().splitlines()
random.seed(0)
sub = random.sample(lines, min(50000, len(lines)))
open('data/synth/synth_ft.jsonl', 'w', encoding='utf-8').write('\n'.join(sub) + '\n')

sys.path.insert(0, '.')
from viettdr.vietchar import decompose_char
tones = Counter()
for line in lines[:20000]:
    for ch in json.loads(line)['text']:
        tones[decompose_char(ch)[2]] += 1
assert len(tones) == 6, 'THIEU THANH DIEU!'
print('synth OK:', len(lines), 'anh, du 6 thanh dieu |', f'{elapsed_h():.2f}h')

In [ ]:
# Cell 4 — RECOGNIZER (prior + bo danh gia): 2 giai doan + eval + LUU NGAY
import os, shutil, torch

PRE = (f'--data data/vintext_words --charset charset_vintext.txt '
       f'--synth-jsonl data/synth/synth_gt.jsonl --synth-dir data/synth '
       f'--synth-aug light --epochs 15 --bs 192 --workers {NPROC} '
       f'--eval-every 5 --val-subset 1500')
FT = (f'--data data/vintext_words --charset charset_vintext.txt '
      f'--synth-jsonl data/synth/synth_ft.jsonl --synth-dir data/synth '
      f'--synth-aug light --epochs 30 --bs 192 --workers {NPROC} '
      f'--eval-every 3 --val-subset 1500 --lr 2e-4')

if not os.path.exists('runs/pre_full/last.pth'):
    !python train.py {PRE} --out runs/pre_full
ck = 'runs/pre_full/best.pth' if os.path.exists('runs/pre_full/best.pth') \
     else 'runs/pre_full/last.pth'
!python train.py {FT} --out runs/full --init-from {ck}

# LUU checkpoint rut gon sang /kaggle/working NGAY (khong doi cuoi phien)
c = torch.load('runs/full/best.pth', map_location='cpu', weights_only=False)
torch.save({'model': c['model'], 'args': c['args'], 'epoch': c['epoch'],
            'best': c['best']}, f'{OUTDIR}/rec_full_slim.pth')
shutil.copy('runs/full/log.csv', f'{OUTDIR}/rec_full_log.csv')
if os.path.exists('runs/pre_full/log.csv'):
    shutil.copy('runs/pre_full/log.csv', f'{OUTDIR}/rec_pre_log.csv')
print('da luu rec_full_slim.pth vao output |', f'{elapsed_h():.2f}h')

!python eval.py --ckpt runs/full/best.pth --data data/vintext_words \
    --split test --charset charset_vintext.txt
shutil.copy('runs/full/eval_test.json', f'{OUTDIR}/rec_full_eval.json')

In [ ]:
# Cell 5 — sinh 120k cap HR/LR + MAT NA DAU (render kep)
import os
if not os.path.exists('data/sr_pairs/pairs_gt.jsonl'):
    !python tools/gen_sr_pairs.py --out data/sr_pairs --count 120000 \
        --fonts fonts --corpus data/vintext_words/train_gt.jsonl \
        assets/general_dict.txt {BG} --workers {NPROC} \
        --bg-cache 40 --tone-balance
n = sum(1 for _ in open('data/sr_pairs/pairs_gt.jsonl', encoding='utf-8'))
print(f'cap SR: {n} | {elapsed_h():.2f}h')

In [ ]:
# Cell 6 — SR: S0 -> S1 -> S2, eval ngay sau tung run, luu ckpt ngay
import os, shutil

SRRUNS = [
    ('sr_s0', '--lam-mask 0.0'),                       # baseline: chi pix
    ('sr_s1', ''),                                      # + mat na dau (1)
    ('sr_s2', '--rec-ckpt runs/full/best.pth '
              '--charset charset_vintext.txt'),         # + recognizer (2)
]
DONE = []
for name, flag in SRRUNS:
    if elapsed_h() > BUDGET_H:
        print(f'### BO QUA {name}: {elapsed_h():.1f}h > {BUDGET_H}h')
        continue
    print('#' * 20, f'{name} ({elapsed_h():.2f}h)', '#' * 20, flush=True)
    !python train_sr.py --data data/sr_pairs --out runs/{name} \
        --epochs 12 --bs 256 --workers {NPROC} {flag}
    if not os.path.exists(f'runs/{name}/best.pth'):
        print(f'[loi] {name} khong co checkpoint')
        continue
    DONE.append(name)
    shutil.copy(f'runs/{name}/best.pth', f'{OUTDIR}/{name}_best.pth')
    shutil.copy(f'runs/{name}/log.csv', f'{OUTDIR}/{name}_log.csv')
    print('=' * 12, name, 'eval synth', '=' * 12, flush=True)
    !python eval_sr.py synth --data data/sr_pairs --ckpt runs/{name}/best.pth \
        --rec-ckpt runs/full/best.pth --charset charset_vintext.txt --limit 3000
    print('=' * 12, name, 'eval real <=16px', '=' * 12, flush=True)
    !python eval_sr.py real --data data/vintext_words --split test --max-h 16 \
        --ckpt runs/{name}/best.pth --rec-ckpt runs/full/best.pth \
        --charset charset_vintext.txt
print(f'SR xong: {DONE} | {elapsed_h():.2f}h')

In [ ]:
# Cell 7 — kiem tra output gon nhe (bai hoc vong truoc)
import os
files = []
for dp, _, fs in os.walk(OUTDIR):
    files += [os.path.join(dp, f) for f in fs]
print(f'output: {len(files)} file')
for f in sorted(files):
    print(f'  {os.path.getsize(f)/1e6:8.1f} MB  {f}')
assert len(files) < 500, 'QUA NHIEU FILE - Kaggle se bo output!'
print(f'\nHOAN TAT | tong {elapsed_h():.2f}h')